# ✨ ComfyUI Colab – LTX Video + VAE aus Google Drive

**Optimiert für:** **LTX Video Node Pack**, **ComfyUI-Manager**, **VideoHelperSuite** und **DemonAlone-Nodes**, keine Modell-Downloads.
Alle **VAE-Modelle** werden direkt aus `/content/drive/MyDrive/ComfyUI/models/vae` (Google Drive) eingebunden und zum Dekodieren der Latents verwendet.
Der Zugriff auf die WebUI läuft über einen **Cloudflare-Tunnel** (`trycloudflare.com`) – kein API-Key nötig.

## Anleitung
1. Kopie anlegen: _File_ > _Save a copy in Drive_
2. GPU aktivieren: _Runtime_ > _Change runtime type_ > _Hardware accelerator_ = **GPU (T4)**
3. Zelle **`⬇️ 1. Cloudflared herunterladen`** ausführen
4. Zelle **`🚩 2. ComfyUI Setup`** ausführen (Drive wird automatisch gemountet)
5. Zelle **`🌐 3. Cloudflare Tunnel starten`** ausführen
6. Öffne die angezeigte `https://xxxx.trycloudflare.com` URL im Browser

> **Hinweis:** Beim ersten Verbinden fragt der Tunnel nach einem Passcode – das ist deine **Colab-IP**:
> `_Runtime_ > _Run before and after_ bzw. `!curl -s ifconfig.me` (wird auch in der Setup-Zelle ausgegeben).
>
> **Tipps:**
> * Drücke **F5**, falls der ComfyUI-Manager nicht richtig lädt.
> * Im Workflow einfach einen **`VAE Load`**-Node verwenden – alle VAEs aus deinem Drive-Ordner stehen in der Liste zur Verfügung.


In [ ]:
#@title ⬇️ 1. Cloudflared herunterladen
#@markdown Lädt den Cloudflare-Tunnel-Client (einmal pro Session nötig).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
print("cloudflared bereit.")


In [ ]:
#@title 🚩 2. ComfyUI Setup
#@markdown ### ▶️ Pfade (nichts ändern, außer dein Drive-Pfad ist ein anderer)
vae_drive_path = '/content/drive/MyDrive/ComfyUI/models/vae' #@param {type:"string"}
drive_root = '/content/drive/MyDrive/ComfyUI' #@param {type:"string"}
Extra_arguments = '--listen 127.0.0.1' #@param {type:"string"}
Clear_Log = True #@param {type:"boolean"}

#####################
import os, sys, glob, time, socket, subprocess, threading, shlex

def iframe_thread(port):
    # Wartet bis ComfyUI laeuft und gibt die IP fuer den Cloudflare-Passcode aus
    while True:
        time.sleep(1)
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if s.connect_ex(('127.0.0.1', port)) == 0:
            break
        s.close()
    import urllib.request
    ip = urllib.request.urlopen('https://ifconfig.me').read().decode('utf8').strip()
    print("\nComfyUI laeuft auf Port %d" % port)
    print("Cloudflare-Passcode (deine Colab-IP): " + ip)

def clear():
    from IPython.display import clear_output
    return clear_output()

# --- Google Drive mounten ---
from google.colab import drive
drive.mount('/content/drive')

# --- VAE-Ordner aus Drive vorbereiten ---
os.makedirs(vae_drive_path, exist_ok=True)
vaes = [f for f in os.listdir(vae_drive_path)
        if f.lower().endswith(('.ckpt', '.safetensors', '.pt', '.bin'))]
print("Gefundene VAE-Modelle in Drive (%d):" % len(vaes))
for v in vaes:
    print("  - " + v)
if not vaes:
    print("Achtung: Kein VAE gefunden! Lege deine .safetensors/.ckpt nach:")
    print("  " + vae_drive_path)

# Input-/Output-Ordner im Drive
image_input_path = os.path.join(drive_root, 'inputs')
image_output_path = os.path.join(drive_root, 'outputs')
os.makedirs(image_input_path, exist_ok=True)
os.makedirs(image_output_path, exist_ok=True)

# --- ComfyUI: nur bei Erstanlauf installieren, danach NUR git pull ---
# LTX Video braucht ein aktuelles ComfyUI (comfy.ldm.lightricks.av_model),
# deshalb Git-Repo statt Release-ZIP -> Update per `git pull`.
subprocess.run("pkill -f 'python.*main.py'", shell=True, stderr=subprocess.DEVNULL)
os.chdir('/content')
ComfyUIRoot = '/content/ComfyUI'
if not os.path.isdir(ComfyUIRoot):
    subprocess.run(['git','clone','-q','--depth','1',
                    'https://github.com/comfyanonymous/ComfyUI.git', ComfyUIRoot], check=True)
    print("ComfyUI installiert.")
else:
    # Bereits vorhanden -> NICHT neu installieren, nur aktualisieren
    print("ComfyUI vorhanden -> git pull (Update auf neueste Version)...")
    r = subprocess.run(['git','-C',ComfyUIRoot,'pull','--quiet'], check=False)
    if r.returncode != 0:
        # Konflikt oder Repo ohne .git -> Working Tree zurücksetzen und erneut pullen
        subprocess.run(['git','-C',ComfyUIRoot,'checkout','.'], check=False)
        subprocess.run(['git','-C',ComfyUIRoot,'pull','--rebase','--quiet'], check=False)

customNodeRoot = os.path.join(ComfyUIRoot, 'custom_nodes')

# Torch ist in Colab bereits vorinstalliert -> nur fehlende Requirements nachziehen
# Bei Installationsfehlern abbrechen, bevor ComfyUI gestartet wird
subprocess.run(['pip','install','-q','-r', os.path.join(ComfyUIRoot,'requirements.txt')], check=True)

# --- Geforderte Nodes installieren (vorhanden -> git pull) ---
def install_custom_node(url):
    repo_name = url.split('/')[-1].replace('.git', '')
    dest = os.path.join(customNodeRoot, repo_name)
    if not os.path.isdir(dest):
        subprocess.run(['git','clone','-q','--depth','1',url,dest], check=False)
    else:
        subprocess.run(['git','-C',dest,'pull','--quiet'], check=False)
    req = os.path.join(dest, 'requirements.txt')
    if os.path.exists(req):
        print(f"Installiere Abhaengigkeiten fuer {repo_name}...")
        subprocess.run(['pip', 'install', '-q', '-r', req], check=False)

install_custom_node('https://github.com/Comfy-Org/ComfyUI-Manager.git')
install_custom_node('https://github.com/Lightricks/ComfyUI-LTXVideo.git')
install_custom_node('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git')
install_custom_node('https://github.com/DemonAlone/DemonAlone-nodes-ComfyUI.git')

# Zeige gefundene VAEs auf Drive
print("VAEs im Drive-Ordner %s:" % vae_drive_path)
for f in sorted(glob.glob(os.path.join(vae_drive_path, '*.*'))):
    if f.lower().endswith(('.safetensors', '.ckpt', '.pt', '.bin')):
        print("  - " + os.path.basename(f))

# --- VAE-Links: Drive -> ComfyUI/models/vae (nur verlinken, kein Download!) ---
vae_local = os.path.join(ComfyUIRoot, 'models', 'vae')
os.makedirs(vae_local, exist_ok=True)
for ext in ('*.safetensors', '*.ckpt', '*.pt', '*.bin'):
    for f in glob.glob(os.path.join(vae_drive_path, ext)):
        link = os.path.join(vae_local, os.path.basename(f))
        if not os.path.exists(link):
            os.symlink(f, link)
            print("Verlinke VAE: " + os.path.basename(f))

# Port ist die einzige autoritative Port-Setting (nicht in Extra_arguments!)
port = 8188

if Clear_Log:
    clear()

# --- ComfyUI starten (im Hintergrund, damit die naechste Zelle den Tunnel oeffnet) ---
threading.Thread(target=iframe_thread, daemon=True, args=(port,)).start()
log = open('/content/comfyui.log', 'w')
subprocess.Popen(
    [sys.executable, 'main.py', '--dont-print-server',
     '--output-directory', image_output_path,
     '--input-directory', image_input_path,
     '--port', str(port)] + shlex.split(Extra_arguments),
    cwd=ComfyUIRoot, stdout=log, stderr=subprocess.STDOUT)
print("ComfyUI startet... bitte gleich Zelle '3. Cloudflare Tunnel' ausfuehren.")
print("Logs: /content/comfyui.log")


In [ ]:
#@title 🌐 3. Cloudflare Tunnel starten
#@markdown Erstellt kostenlos eine öffentliche URL (trycloudflare.com) – kein Account/Key nötig.
import atexit, requests, subprocess, time, re, os
from random import randint
from threading import Thread
from queue import Queue

def cloudflared(port, metrics_port, output_queue):
    try:
        proc = subprocess.Popen(
            ['/content/cloudflared', 'tunnel', '--no-autoupdate',
             '--url', f'http://127.0.0.1:{port}',
             '--metrics', f'127.0.0.1:{metrics_port}'],
            stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        atexit.register(lambda p=proc: p.terminate())
        attempts, tunnel_url = 0, None
        while attempts < 10 and not tunnel_url:
            attempts += 1
            time.sleep(3)
            try:
                tunnel_url = re.search(r"(?P<url>https?://[^\s]+\.trycloudflare\.com)",
                    requests.get(f'http://127.0.0.1:{metrics_port}/metrics', timeout=5).text).group("url")
            except Exception:
                pass
        if not tunnel_url:
            raise Exception("Can't connect to Cloudflare Edge")
        output_queue.put(('ok', tunnel_url))
    except Exception as e:
        output_queue.put(('error', e))

port = 8188
output_queue, metrics_port = Queue(), randint(8100, 9000)
thread = Thread(target=cloudflared, args=(port, metrics_port, output_queue))
thread.start()
thread.join()
kind, result = output_queue.get()
if kind == 'error':
    raise result
tunnel_url = result
os.environ['webui_url'] = tunnel_url
print("ComfyUI URL: " + tunnel_url)
print("Passcode beim ersten Oeffnen = deine Colab-IP (!curl -s ifconfig.me)")


In [ ]:
#@title 🔗 3b. Pinggy Tunnel starten (Alternative zu Cloudflare)
#@markdown Gratis-Tunnel via SSH – keine Installation, kein Key. Nur eine der beiden Tunnel-Zellen ausfuehren!
import subprocess, threading, time, os

port = 8188
pinggy_cmd = (f"ssh -p 443 -R0:127.0.0.1:{port} "
              "-o StrictHostKeyChecking=no -o ServerAliveInterval=30 "
              "DphNQlI9bNG@free.pinggy.io")

def pinggy_loop():
    # Aehnlich der Windows-Batch-Schleife: verbinde erneut, wenn SSH beendet
    while True:
        p = subprocess.Popen(pinggy_cmd.split(),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line.rstrip())
            if 'https://' in line:   # Pinggy druckt die Tunnel-URL in einer Zeile
                url = line.strip().split()[-1]
                os.environ['webui_url'] = url
                print("Pinggy URL: " + url)
        p.wait()
        print("SSH getrennt -> neuer Verbindungsversuch in 10s...")
        time.sleep(10)

threading.Thread(target=pinggy_loop, daemon=True).start()
print("Pinggy-Tunnel startet im Hintergrund – URL erscheint gleich hier.")

In [ ]:
#@title 📂 4. Ausgaben als ZIP sichern (optional)
#@markdown ### ▶️ Zipping:
zip_prefix = '' #@param {type:"string"}
zip_subfolder = '' #@param {type:"string"}

import os, zipfile, glob
from datetime import datetime

base_path = os.path.join(drive_root, 'outputs')
if zip_subfolder:
    base_path = os.path.join(base_path, zip_subfolder)
os.makedirs(base_path, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_path = os.path.join(base_path, f'{zip_prefix or "ComfyUI_"}files_{timestamp}.zip')

files = sorted(set(
    glob.glob(os.path.join(base_path, f'{zip_prefix}*.mp4')) +
    glob.glob(os.path.join(base_path, f'{zip_prefix}*.png')) +
    glob.glob(os.path.join(base_path, f'{zip_prefix}*.webm'))))
files = [f for f in files if os.path.isfile(f)]
if not files:
    print(f"Keine Dateien mit Praefix '{zip_prefix}' in {base_path}")
else:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in files:
            zf.write(f, os.path.basename(f))
            print("Added " + os.path.basename(f))
    print("ZIP erstellt: " + zip_path)
